# 01 data measurement

### Question

Data infoment survey

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import Phylo
from openpyxl import load_workbook

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]

start = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in [start, *start.parents]
     if (candidate / "data" / "106bac.parquet").exists()),
    None,
)

DATA = ROOT / "data"
FILES = {
    "neural": DATA / "106bac.parquet",
    "lcms_fold_change": DATA / "matrix.xlsx",
    "lcms_raw": DATA / "metabolism_raw_data.xlsx",
    "taxonomy": DATA / "GM300_bacteria_species_summary.xlsx",
    "tree": DATA / "16S.aln.trim.fa.treefile",
    "current_samples": DATA / "current_samples.xlsx",
}

file_inventory = pd.DataFrame(
    {
        "exists": {name: path.exists() for name, path in FILES.items()},
        "size_MiB": {name: path.stat().st_size / 2**20 for name, path in FILES.items()},
    }
).round(2)
file_inventory

In [ ]:
df = pd.read_parquet(FILES["neural"])


DROP_NEURONS = {"AFDL", "AFDR", "BAGL", "BAGR", "CEPR", "URXR"}
df = df.loc[~df["neuron"].isin(DROP_NEURONS)].reset_index(drop=True)

df["sample_id"] = df["stim_name"].str.extract(r"^(A\d{3})", expand=False)
basic_summary = pd.Series(
    {
        "rows": len(df),
        "columns": df.shape[1],
        "memory_MiB": df.memory_usage(deep=True).sum() / 2**20,
        "null_cells": int(df.isna().sum().sum()),
        "nonfinite_delta_F_over_F0": int((~np.isfinite(df["delta_F_over_F0"].to_numpy())).sum()),
        "unique_neurons": df["neuron"].nunique(),
        "unique_stimulus_codes": df["stimulus"].nunique(),
        "unique_dates": df["date"].nunique(),
        "unique_worm_labels": df["worm_key"].nunique(),
        "unique_time_points": df["time_point"].nunique(),
    },
    name="value",
)
display(basic_summary.to_frame())
display(df.dtypes.rename("dtype").to_frame())
display(df.head(8))

stimulus这一列可以完全删除，目前没有意义，真正有意义的是stim_name

后续使用的neuron为：AWC AWB AWA ADF ASK ASE ASG ASI ADL ASH ASJ

共有 22 个 neuron label(先过滤掉了heat sensory以及gas sensory neuron)、106 个 stimulus code、9 个日期、8 个重复使用的 `w1…w8` 标签，以及 45 个 timepoint

### Observation

106 个内部 `stimulus` code 与 106 个 Axxx 样本号严格一一对应，所有 `stim_name` 都以 `stationary` 结尾。后续以 `sample_id` 作为细菌身份，保留 `stimulus` 仅用于追溯原表。

In [ ]:
wrong_row_key = ["worm_key", "sample_id", "segment_index", "neuron", "time_point"]
correct_row_key = ["date", *wrong_row_key]

key_check = pd.Series(
    {
        "duplicate_rows_without_date": int(df.duplicated(wrong_row_key).sum()),
        "duplicate_rows_with_date": int(df.duplicated(correct_row_key).sum()),
        "worm_labels": df["worm_key"].nunique(),
        "date_x_worm_individuals": df.groupby(["date", "worm_key"]).ngroups,
        "dates_per_worm_label_min": int(df.groupby("worm_key")["date"].nunique().min()),
        "dates_per_worm_label_max": int(df.groupby("worm_key")["date"].nunique().max()),
    },
    name="value",
)
display(key_check.to_frame())
display(pd.crosstab(df["date"], df["worm_key"]).gt(0).astype(int))

0313 w2 no signal
0331 w2 bad expression
0410 w3 bad immobilization
0414 w3 no signal
0429 w4 not processed
0601 w5 not processed

In [ ]:
# worm_key labels are reused across dates; each (date, worm_key) pair is one individual.
individual_key = ["date", "worm_key"]
trial_key = [*individual_key, "sample_id", "segment_index"]
trace_key = [*trial_key, "neuron"]
row_key = [*trace_key, "time_point"]

trials = df[trial_key + ["stimulus", "stim_name"]].drop_duplicates()
traces = df[trace_key].drop_duplicates()
trace_time = df.groupby(trace_key)["time_point"].agg(["size", "nunique", "min", "max"])

trial_contract = pd.Series(
    {
        "individuals": df.groupby(individual_key).ngroups,
        "operational_trials": len(trials),
        "neuron_traces": len(traces),
        "duplicate_rows_under_full_key": int(df.duplicated(row_key).sum()),
        "min_rows_per_trace": int(trace_time["size"].min()),
        "max_rows_per_trace": int(trace_time["size"].max()),
        "min_unique_timepoints_per_trace": int(trace_time["nunique"].min()),
        "max_unique_timepoints_per_trace": int(trace_time["nunique"].max()),
    },
    name="value",
)
display(trial_contract.to_frame())

segments_per_sample_individual = trials.groupby(
    [*individual_key, "sample_id"]
).size().value_counts().sort_index().rename("n_sample_individuals")
display(segments_per_sample_individual.to_frame().rename_axis("trials_per_sample_individual"))

### Question

神经、LCMS fold change、原始 LCMS、分类信息和 16S tree 的样本集合如何对齐？

In [ ]:
matrix = pd.read_excel(FILES["lcms_fold_change"], index_col=0)
matrix.index = matrix.index.astype(str).str.strip()
matrix_ids = set(matrix.index)

species = pd.read_excel(
    FILES["taxonomy"], sheet_name="Axxx_species_mapping"
)
species_ids = set(species["AID"].dropna().astype(str).str.strip())

tree = Phylo.read(FILES["tree"], "newick")
tree_ids = {terminal.name for terminal in tree.get_terminals() if terminal.name}

raw_book = load_workbook(FILES["lcms_raw"], read_only=True, data_only=False)
raw_header = [cell.value for cell in next(raw_book["all"].iter_rows(min_row=1, max_row=1))]
raw_ids = {
    str(value).strip()
    for value in raw_header
    if re.fullmatch(r"A\d{3}", str(value).strip())
}
raw_sheet_shapes = {
    ws.title: (ws.max_row - 1, ws.max_column) for ws in raw_book.worksheets
}
raw_book.close()

current = pd.read_excel(FILES["current_samples"])
current_ids = set(current.iloc[:, 0].dropna().astype(str).str.strip())
neural_ids = set(df["sample_id"].unique())

source_sets = {
    "neural": neural_ids,
    "matrix_fold_change": matrix_ids,
    "species_mapping": species_ids,
    "16S_tree": tree_ids,
    "raw_LCMS_all_sheet": raw_ids,
    "current_samples": current_ids,
}
coverage = pd.DataFrame(
    {
        "n_Axxx": {name: len(ids) for name, ids in source_sets.items()},
        "neural_missing_from_source": {
            name: len(neural_ids - ids) for name, ids in source_sets.items()
        },
    }
)
display(coverage)
display(pd.Series(raw_sheet_shapes, name="(data_rows, columns)").to_frame())
print("matrix shape:", matrix.shape, "| matrix NaN cells:", int(matrix.isna().sum().sum()))
print("matrix ↔ species set differences:", sorted(matrix_ids ^ species_ids))
print("tree ↔ species set differences:", sorted(tree_ids ^ species_ids))
print("raw-only Axxx relative to matrix:", sorted(raw_ids - matrix_ids))

### Observation

106 个神经样本全部能在 fold-change matrix、种属表、16S tree、原始 LCMS 和 `current_samples` 中找到。`matrix.xlsx` 是 299 samples × 380 metabolites，数值区无 NaN；它与 species mapping 和 tree 的 299 个 Axxx 集合完全一致。原始 LCMS `all` sheet 有 380 个数据行和 319 个 Axxx columns，多出的 20 个样本为 A050–A058、A088–A092、A225、A250、A306、A316–A318。它们为何未进入 299-sample matrix 需要查看 LCMS/QC 生成记录。

### Question

106 个细菌目前各有多少 trial、覆盖多少个 individual、跨多少日期？

In [ ]:
sample_individuals = trials[[*individual_key, "sample_id"]].drop_duplicates()
per_sample = trials.groupby("sample_id").agg(
    n_trials=("segment_index", "size"),
    n_dates=("date", "nunique"),
    dates=("date", lambda x: ", ".join(sorted(x.unique()))),
)
per_sample["n_individuals"] = sample_individuals.groupby("sample_id").size()
per_sample = per_sample[["n_trials", "n_individuals", "n_dates", "dates"]]

display(per_sample[["n_trials", "n_individuals", "n_dates"]].describe().round(2))
display(pd.concat({"lowest trial count": per_sample.nsmallest(10, "n_trials"),
                   "highest trial count": per_sample.nlargest(10, "n_trials")}))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.histplot(per_sample["n_trials"], discrete=True, ax=axes[0])
axes[0].set(title="Trials per bacterium", xlabel="operational trials")
sns.scatterplot(data=per_sample, x="n_individuals", y="n_trials", hue="n_dates", palette="viridis", ax=axes[1])
axes[1].set(title="Trial count and individual coverage", xlabel="individuals", ylabel="trials")
plt.tight_layout()
plt.show()

### Observation

每个细菌有 17–59 个 trials（中位数 22），覆盖 4–14 个 individuals。106 个中有 100 个只在一个日期出现；跨日期样本因为增加了 individual，通常也有更多 trials。

2,696 个 trials 在目前意义上不能当成完全独立重复，worm date可能都会对其产生影响，可以根据影响的大小决定后续的分析思路。

### Question

每天测了哪些细菌？哪些细菌跨日期重复？

In [ ]:
date_summary = trials.groupby("date").agg(
    n_trials=("segment_index", "size"),
    n_samples=("sample_id", "nunique"),
    n_individuals=("worm_key", "nunique"),
)
samples_per_date = trials.groupby("date")["sample_id"].agg(
    lambda x: ", ".join(sorted(x.unique()))
)
date_table = date_summary.join(samples_per_date.rename("sample_ids"))

cross_date = per_sample.loc[per_sample["n_dates"] > 1].sort_index()
display(date_table)
display(cross_date)

fig, ax = plt.subplots(figsize=(10, 3.6))
date_summary[["n_trials", "n_samples", "n_individuals"]].plot.bar(ax=ax)
ax.set(title="Measurement coverage by date", xlabel="date", ylabel="count")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### Observation

数据覆盖 9 个日期。每天有 11–13 个细菌、4–7 个 `(date, worm)` individuals 和 239–413 个 trials。跨日期重复仅有 6 个样本：A011、A013、A014、A024、A025、A044；每个都出现在两个日期。日期与细菌集合高度绑定，因此 date/batch 与 bacteria identity 存在明显混杂，后续必须保留日期层级。

### Question

每个 trial 有哪些 neuron？neuron panel 在同一 individual 内是否稳定？

In [ ]:
trial_neuron_sets = traces.groupby(trial_key)["neuron"].agg(lambda x: tuple(sorted(x)))
trial_neuron_n = trial_neuron_sets.str.len()
individual_neuron_sets = traces.groupby(individual_key)["neuron"].agg(lambda x: tuple(sorted(x.unique())))
sets_per_individual = trial_neuron_sets.groupby(level=individual_key).nunique()

display(trial_neuron_n.describe().round(2).to_frame("neurons_per_trial"))
display(trial_neuron_n.value_counts().sort_index().rename("n_trials").to_frame())
print("unique neuron-set count within each individual:")
display(sets_per_individual.value_counts().sort_index().rename("n_individuals").to_frame())

individual_presence = (
    traces[[*individual_key, "neuron"]]
    .drop_duplicates()
    .assign(present=1)
    .pivot_table(index=individual_key, columns="neuron", values="present", fill_value=0)
)
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(individual_presence, cmap=["#f1f1f1", "#2c7fb8"], cbar=False, ax=ax)
ax.set_title("Neuron presence by individual (date × worm)")
ax.set_xlabel("neuron")
ax.set_ylabel("individual")
plt.tight_layout()
plt.show()

### Observation

每个 trial 有 8–22 个 neurons（中位数 19）。49 个 individuals 中，每一个 individual 内的 neuron set 都完全不变；同一个individual的trial neuron数目基本一致。

### Question

数值缺失、timepoint 缺失和 neuron 缺失分别是多少？

In [ ]:
individual_union = traces.groupby(individual_key)["neuron"].agg(set)
individual_trial_n = trials.groupby(individual_key).size()
expected_trace_slots = sum(
    len(individual_union.loc[key]) * int(individual_trial_n.loc[key])
    for key in individual_union.index
)

missing_summary = pd.Series(
    {
        "NaN rate among stored ΔF/F0 values": df["delta_F_over_F0"].isna().mean(),
        "non-finite rate among stored ΔF/F0 values": (~np.isfinite(df["delta_F_over_F0"].to_numpy())).mean(),
        "incomplete trace rate (expected timepoints 0–44)": (trace_time["nunique"] != 45).mean(),
        "duplicate row rate under full key": df.duplicated(row_key).mean(),
        "missing neuron-trace rate within individual-specific panels": 1 - len(traces) / expected_trace_slots,
    },
    name="rate",
)
display(missing_summary.map(lambda x: f"{x:.3%}").to_frame())

neuron_coverage = traces.groupby("neuron").agg(
    n_traces=("segment_index", "size"),
    n_individuals=("date", "size"),
)
neuron_coverage["n_individuals"] = (
    traces[[*individual_key, "neuron"]].drop_duplicates().groupby("neuron").size()
)
neuron_coverage["fraction_of_all_trials"] = neuron_coverage["n_traces"] / len(trials)
display(neuron_coverage.sort_values("fraction_of_all_trials").round(3))

将ASK ADL ASI AWA AWB ASG ADF ASH ASJ这9个functional symmetry的neuron的L/R看作一个，再次统计fraction_of_all_trials

In [ ]:
POOLED = {
    "ASK": ["ASKL", "ASKR"],
    "ADL": ["ADLL", "ADLR"],
    "ASI": ["ASIL", "ASIR"],
    "AWA": ["AWAL", "AWAR"],
    "AWB": ["AWBL", "AWBR"],
    "ASG": ["ASGL", "ASGR"],
    "ADF": ["ADFL", "ADFR"],
    "ASH": ["ASHL", "ASHR"],
    "ASJ": ["ASJL", "ASJR"],
}
neuron_to_class = {n: cls for cls, members in POOLED.items() for n in members}

pooled = traces.copy()
pooled["class"] = pooled["neuron"].map(neuron_to_class).fillna(pooled["neuron"])

class_trials = pooled[[*trial_key, "class"]].drop_duplicates()
class_coverage = class_trials.groupby("class").agg(n_trials=("segment_index", "size"))
class_coverage["n_individuals"] = (
    class_trials[[*individual_key, "class"]].drop_duplicates().groupby("class").size()
)
class_coverage["fraction_of_all_trials"] = class_coverage["n_trials"] / len(trials)

display(class_coverage.sort_values("fraction_of_all_trials").round(3))


将对称neuron填充之后，fraction of all trials基本都在0.8以上，但怎么合并需要思考，简单将L与R当成两个trial显然不合理，但我目前是这样做的

### Question

`start_time` / `end_time` 对应哪些 timepoints？当前文件能否确认秒数和端点包含规则？

In [ ]:
time_contract = pd.Series(
    {
        "time_point_min": df["time_point"].min(),
        "time_point_max": df["time_point"].max(),
        "n_time_points": df["time_point"].nunique(),
        "unique_start_time": df["start_time"].nunique(),
        "start_time_value": df["start_time"].iloc[0],
        "unique_end_time": df["end_time"].nunique(),
        "end_time_value": df["end_time"].iloc[0],
        "sampling_interval_column_present": any(
            name in df.columns for name in ["sampling_interval", "dt", "time_seconds"]
        ),
    },
    name="value",
)
display(time_contract.to_frame())

ONSET = int(df["start_time"].iloc[0])
OFFSET = int(df["end_time"].iloc[0])

# 工作假设：end_time 是 exclusive boundary。修改这一行即可测试另一种约定。
END_IS_EXCLUSIVE = True
phase_table = pd.DataFrame(
    {
        "phase": ["baseline", "stimulus (working assumption)", "post-stimulus"],
        "timepoints": [
            list(range(0, ONSET)),
            list(range(ONSET, OFFSET if END_IS_EXCLUSIVE else OFFSET + 1)),
            list(range(OFFSET if END_IS_EXCLUSIVE else OFFSET + 1, 45)),
        ],
    }
)
phase_table["n_points"] = phase_table["timepoints"].str.len()
display(phase_table)

所有行的 onset marker 都是 5，offset marker 都是 15。表本身没有采样间隔或绝对秒数列，因此只能确认索引，不能独立证明 1 point = 1 s。当前工作约定把 `end_time` 视为 exclusive boundary：baseline 0–4（5 点）、stimulus 5–14（10 点）、post 15–44（30 点）

### Question

能否从已归一化的 traces 反推当前 baseline 使用了哪些点？

In [ ]:
baseline_rows = df[df["time_point"] < ONSET]
baseline_trace = baseline_rows.groupby(trace_key)["delta_F_over_F0"].agg(
    baseline_mean="mean",
    baseline_sd="std",
    baseline_range=lambda x: x.max() - x.min(),
)

display(
    baseline_trace[["baseline_mean", "baseline_sd", "baseline_range"]]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .round(5)
)
print("max |mean ΔF/F0 over timepoints 0–4|:", baseline_trace["baseline_mean"].abs().max())

### Observation

48,701 条 trace 在 timepoints 0–4 上的平均 $\Delta F/F_0$ 都等于 0（最大浮点误差约 $3.6	imes10^{-16}$）。这强烈支持 0–4 是归一化所用 baseline window，并且归一化使该窗口均值归零。

### Question

每个 trial 的 baseline fluctuation 多大，是否集中在某些日期或 individuals？

In [ ]:
trial_baseline = baseline_trace.groupby(level=trial_key).agg(
    n_neurons=("baseline_sd", "size"),
    median_neuron_sd=("baseline_sd", "median"),
    p90_neuron_sd=("baseline_sd", lambda x: x.quantile(0.9)),
    max_neuron_sd=("baseline_sd", "max"),
)

display(
    trial_baseline.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(4)
)
display(trial_baseline.nlargest(12, "median_neuron_sd").round(4))

plot_baseline = trial_baseline.reset_index()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(plot_baseline["median_neuron_sd"], bins=40, ax=axes[0])
axes[0].axvline(plot_baseline["median_neuron_sd"].quantile(0.99), color="C3", ls="--", label="99th percentile")
axes[0].legend()
axes[0].set(title="Trial baseline fluctuation", xlabel="median neuron baseline SD")
sns.boxplot(data=plot_baseline, x="date", y="median_neuron_sd", color="#8ecae6", ax=axes[1])
axes[1].tick_params(axis="x", rotation=45)
axes[1].set(title="Baseline fluctuation by date", xlabel="date", ylabel="median neuron baseline SD")
plt.tight_layout()
plt.show()

### Observation

trial-level median neuron baseline SD 的中位数为 0.0563，95th percentile 为 0.0954，99th percentile 为 0.1211，最大值 0.2571。最高的 6 个 trial 全来自 2026-03-31 的 w4

### Question

baseline SD 最大的 traces 是单点跳变，还是持续波动？

In [ ]:
worst_baseline = baseline_trace.nlargest(8, "baseline_sd").reset_index()
display(worst_baseline.round(4))

worst_baseline_curves = df.merge(worst_baseline[trace_key], on=trace_key, how="inner")
g = sns.FacetGrid(
    worst_baseline_curves,
    col="neuron",
    col_wrap=4,
    hue="sample_id",
    sharey=False,
    height=2.5,
    aspect=1.25,
)
g.map_dataframe(sns.lineplot, x="time_point", y="delta_F_over_F0", estimator=None)
for ax in g.axes.flat:
    ax.axvspan(0, ONSET - 1, color="#bbbbbb", alpha=0.18)
    ax.axvline(ONSET, color="C2", ls="--", lw=1)
    ax.axvline(OFFSET, color="C3", ls="--", lw=1)
g.set_axis_labels("time_point", "ΔF/F0")
g.fig.suptitle("Traces with the largest baseline SD", y=1.03)
plt.show()

### Observation

最大的单条 baseline SD 为 1.693（2026-04-14, w2, A013, segment 19, ASJR）。高 baseline SD 涉及 ASJR、ASJL、ASIL、ASEL、AWCON 等不同 neurons 和多个 individuals；原始曲线可用于区分宽幅波动、漂移和个别点跳变。现阶段只标记、追溯，不删除。

### Question

是否存在极端 $\Delta F/F_0$ traces，它们集中在哪些 neurons？

In [ ]:
trace_stats = df.groupby(trace_key)["delta_F_over_F0"].agg(
    trace_min="min",
    trace_max="max",
    trace_mean="mean",
    trace_sd="std",
)
trace_stats["max_abs"] = trace_stats[["trace_min", "trace_max"]].abs().max(axis=1)
trace_stats = trace_stats.join(baseline_trace)

quantiles = trace_stats["max_abs"].quantile([0.5, 0.9, 0.95, 0.99, 0.999, 1.0])
threshold_counts = []
for threshold in [2, 3, 5, 8]:
    selected = trace_stats[trace_stats["max_abs"] >= threshold].reset_index()
    threshold_counts.append(
        {
            "|ΔF/F0| threshold": threshold,
            "n_traces": len(selected),
            "AWCON_traces": int((selected["neuron"] == "AWCON").sum()),
        }
    )

top_extreme = trace_stats.nlargest(15, "max_abs").reset_index()
display(quantiles.rename("max_abs").to_frame().round(4))
display(pd.DataFrame(threshold_counts))
display(top_extreme.round(4))

fig, ax = plt.subplots(figsize=(8, 3.8))
sns.histplot(trace_stats["max_abs"], bins=100, ax=ax)
ax.set_yscale("log")
ax.axvline(quantiles.loc[0.99], color="C3", ls="--", label="99th percentile")
ax.set(title="Distribution of per-trace maximum |ΔF/F0|", xlabel="maximum |ΔF/F0|", ylabel="trace count (log scale)")
ax.legend()
plt.tight_layout()
plt.show()

per-trace 最大绝对幅度的 99th percentile 为 2.479，99.9th percentile 为 7.548，最大值为 10.028。共有 118 条 trace 达到 $|\Delta F/F_0|\ge5$，其中 94 条是 AWCON；达到 8 的 35 条中 33 条是 AWCON。极值明显有 neuron 结构，不能用单一全局阈值自动当作坏数据。

## 本轮 measurement checkpoint

已经可以直接从文件确认：

- 106 个神经刺激样本，49 个 individuals（由 `date × worm_key` 唯一确定），2,696 个 operational trials，48,701 条完整 45-point traces。
- 不同 date 下的同名 `worm_key` 对应不同individual；因此 `(date, worm_key)` 是 individual key，忽略 date 会把不同 individual 错误合并。
- neuron panel 在每个 individual 内恒定，跨 individual 不同。    
- onset/offset markers 为 5/15；0–4 的 baseline 均值对每条 trace 精确归零。
- baseline fluctuation 存在 individual-level 集中；极端幅度主要集中在 AWCON，并呈完整连续动态。

补充事实：
1. 每个 `time_point` 严格等于 1 s，每个point为一个完整volume。
2. `end_time=15` end_time比较模糊，目前认为刺激已经在15结束
3. `segment_index` 为对一个individual施加的第几次刺激，是相对所有刺激而言的
4. $\Delta F/F_0$ 的 $F_0$ 是根据baseline区域样本点的曲线拟合得到的，具体而言，提供指数拟合、双指数拟合以及直线拟合三种function，采用AIC选择最好的，得到F_0
6. 原始 LCMS 中 20 个 raw-only samples 为什么未进入 299-sample fold-change matrix？原因目前未知，且目前而言并不重要